# Knee MRI: a self-supervised backbone, adapted

**Public LB 0.883.** Five fine-tuned DINOv2 ViT-B/14 models, no external predictions blended in.

One switch at the top picks between two published weight sets — a 224px run that scores **0.866 in ~3 h**, and a 336px run that scores **0.883 in ~4 h**. Which one you want depends on which board you are aiming at.

This notebook is the inference half of a two-part system. The training half runs off-Kaggle; what is published here is the complete scoring path, the reasoning behind each choice, and — more usefully — the decision rule that nearly made me throw away the better of the two models.

---

## The shape of the problem

Twelve binary findings per study — two ligaments, two menisci, three osteoarthritis compartments, effusion, synovitis, Baker's cyst, contusion, fracture — scored as the unweighted mean of twelve ROC AUCs.

The decisive fact is in `train.csv`. Of 4 407 training studies, **58 carry the twelve labels**. The other 4 349 carry a radiology report instead, and the test set carries no report at all. So the pipeline has exactly one possible shape: turn reports into targets, train a pure imaging model against those targets, and discard the text before prediction time.

Two consequences worth internalising before reading further.

**Only rank order matters.** AUC is invariant under any strictly increasing transform, so calibration buys nothing and thresholds buy nothing.

**Every label costs the same.** A finding left at chance forfeits `(M − 0.5)/12` of the final score regardless of how well the other eleven do. That makes the weakest label, not the strongest, the thing worth staring at.

---

## Where the targets came from (summary)

The reports arrive in nine languages. A local LLM (Qwen3-14B-AWQ, served with vLLM under a fixed JSON schema) reads each report into a closed vocabulary of discrete findings — `complete_tear`, `degeneration_no_tear`, `absent_explicit`, severity grades — and a deterministic Python layer maps those states onto probabilities. Two layers rather than one, because asking a model for calibrated numbers directly is asking it to do something it is bad at, while asking it to pick from a fixed vocabulary is something it is good at.

The output is **soft targets, not binaries**. An effusion described as mild is empirically positive about 45% of the time, so it receives 0.45 rather than 1. Each class also carries a confidence weight that scales its loss contribution.

Measured against the 58 annotated studies, the reader reaches macro AUC **0.881**.

And here is the finding that shaped everything downstream: **those 58 studies were annotated by radiologists reading the images, not the reports.** Report and image disagree irreducibly maybe 10–15% of the time. The report reader therefore has a ceiling around 0.88–0.90 — and the imaging model, which sees what the annotators saw, can and should beat its own teacher.

That last sentence is not a flourish. It is the reason the validation section below is arranged the way it is, and the reason I got a decision wrong.

---

## Results, in order

| system | OOF (soft labels) | gold-58 | public LB | runtime |
|---|---:|---:|---:|---:|
| frozen DINOv2 + trained head | 0.682 | 0.771 | 0.776 | ~2 h |
| Stage A — 5-fold fine-tune @224px | 0.717 | 0.824 | **0.866** | ~3 h |
| Stage B — 5-fold fine-tune @336px | 0.722 | — | **0.883** | ~4 h |

Note what those columns do relative to each other. Between Stage A and Stage B, OOF moves **+0.005** and the leaderboard moves **+0.017**. Between the frozen baseline and Stage A, OOF moves +0.035 and the leaderboard moves +0.090. The local metric computed on 4 407 studies systematically *understates* real gains — and §7 explains why that is structural rather than bad luck.

## 1. Configuration

Set `STAGE` and everything else follows. The two weight sets are architecturally identical — same backbone, same head, same training recipe — and differ only in input resolution and in the preprocessing that fed them.

| dataset | resolution | LB | runtime | pick it for |
|---|---|---:|---:|---|
| `rsna-ft-a` | 224px | 0.866 | ~3 h | efficiency prize |
| `rsna-ft-b` | 336px | 0.883 | ~4 h | main leaderboard |

Also attach `rsna-wheels` (offline decoders) and the competition data. No pretrained-backbone dataset is needed: each checkpoint holds a complete backbone *and* its head, so `timm` is instantiated with `pretrained=False` and never touches the network.

`FOLDS` is the second knob. Each fold is an independent model and inference cost scales with how many you load.

In [ ]:
import os, glob, time, collections
import numpy as np, pandas as pd

T0 = time.time()

COMP     = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
D_WHEELS = '/kaggle/input/datasets/sadamtorres/rsna-wheels'

# --- parameters ---
STAGE = 'B'                 # 'A' = 224px, LB .866, ~3h | 'B' = 336px, LB .883, ~4h
FOLDS = [0, 1, 2, 3, 4]     # drop folds to trade AUC for runtime
FEAT  = 'both'              
K     = 32                  # slices sampled per series

D_FT, IMG = {'A': ('/kaggle/input/datasets/sadamtorres/rsna-ft-a', 224),
             'B': ('/kaggle/input/datasets/sadamtorres/rsna-ft-b', 336)}[STAGE]

CLASSES = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
           'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
PREV = dict(zip(CLASSES, [0.196,0.103,0.429,0.210,0.228,0.157,0.307,0.410,0.407,0.242,0.216,0.223]))

TEST_DIR = f'{COMP}/test_series'
test_studies = sorted(os.listdir(TEST_DIR))
print(f'stage {STAGE} @ {IMG}px | {len(test_studies)} studies | {len(FOLDS)} models')

## 2. Offline dependencies, and one thing worth checking

Internet is off, so decoders install from wheels. `pylibjpeg` and `python-gdcm` are insurance rather than necessity: every sampled training file was Explicit VR Little Endian, uncompressed — but the test set is not visible, and a study that fails to decode is a study scored at prevalence.

The census below reads one header per study across the first 200 and counts transfer syntaxes. It costs about a minute and tells you in minute one, rather than minute four hundred, whether the test set is shaped like the training set.

In [ ]:
!pip install --no-index --find-links {D_WHEELS} \
    pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm timm -q

import pydicom, cv2, torch, timm
import torch.nn as nn
print('pydicom', pydicom.__version__, '| timm', timm.__version__, '| cuda', torch.cuda.is_available())

ts = collections.Counter()
for s in test_studies[:200]:
    f = glob.glob(f'{TEST_DIR}/{s}/*/*.dcm')
    if f:
        ts[str(pydicom.dcmread(f[0], stop_before_pixels=True).file_meta.TransferSyntaxUID.name)] += 1
print(ts)

## 3. Geometry: which knee, and which way round

This is the part of the pipeline that repays attention most, and the part most likely to be silently wrong.

**Four of the twelve findings distinguish medial from lateral.** Medial and lateral meniscus, medial and lateral OA. If half your studies are left knees and you do nothing about it, the model sees mirrored anatomy half the time and those four labels are learned as noise. Worse, the failure is invisible: nothing crashes, the loss still goes down, and you lose a third of the score without a symptom.

So laterality has to be recovered per study. The obvious route fails: `ImageLaterality` is absent, and `Laterality` is present but empty in roughly 72% of studies. Geometry answers instead — the sign of the x coordinate of the image **centre** in patient space.

The word *centre* is doing real work there. `ImagePositionPatient` gives the position of the first voxel — a corner, not a middle. Recovering the centre means walking half the image width and height along the direction cosines:

```
centre = IPP + row_cosine · col_spacing · (Columns/2)
             + col_cosine · row_spacing · (Rows/2)
```

Skipping that correction drops agreement with the (rare) populated tag from **96.9% to 58.8%** — from usable to barely better than a coin flip. Within a 25 mm dead zone around the origin the tag wins when present; outside it, geometry wins.

Once laterality is known, everything is canonicalised to *as if it were a right knee*: coronal and axial series from left knees are flipped horizontally, and sagittal slice order is reversed by laterality so that **slice 0 of every sagittal series is the most medial**. That last one is not just tidiness — it makes the slice index anatomically meaningful, and the model exploits it (see §4).

Slice ordering deserves its own note. `InstanceNumber` is not trustworthy for this; slices are sorted by projecting `ImagePositionPatient` onto the slice normal, which is the physical answer rather than a filing convention.

One more consequence, carried through to training: **no horizontal-flip augmentation, ever.** Having spent this much effort making left and right consistent, flipping at random would destroy exactly the information that four labels depend on.

In [ ]:
PLANES = {'sagittal': 0, 'coronal': 1, 'axial': 2}
DEAD_ZONE_MM = 25.0
IM_MEAN = np.array([0.485,0.456,0.406], np.float32)
IM_STD  = np.array([0.229,0.224,0.225], np.float32)

def plane_from_iop(iop):
    """Plane = axis the slice normal points along."""
    row, col = np.array(iop[:3], float), np.array(iop[3:], float)
    n = np.cross(row, col)
    return ['sagittal','coronal','axial'][int(np.argmax(np.abs(n)))], row, col

def center_x(ds):
    """x of the image CENTRE in patient space.
    IPP is a corner; without walking half the extent along the direction
    cosines, agreement with the laterality tag falls 96.9% -> 58.8%."""
    ipp = np.array(ds.ImagePositionPatient, float)
    row = np.array(ds.ImageOrientationPatient[:3], float)
    col = np.array(ds.ImageOrientationPatient[3:], float)
    py, px = map(float, ds.PixelSpacing)
    c = ipp + row * px * (int(ds.Columns) / 2) + col * py * (int(ds.Rows) / 2)
    return float(c[0])

def study_laterality(headers):
    """Geometry decides outside a 25 mm dead zone; inside it, the tag decides."""
    xs = [center_x(ds) for ds in headers if hasattr(ds, 'ImagePositionPatient')]
    xm = float(np.mean(xs)) if xs else 0.0
    tags = {str(getattr(ds, k, '') or '').upper()[:1]
            for ds in headers for k in ('ImageLaterality','Laterality')} - {''}
    tag = tags.pop() if len(tags) == 1 else None
    if abs(xm) < DEAD_ZONE_MM and tag in ('L','R'):
        return tag
    return 'L' if xm > 0 else 'R'

def crop_background(vol_u8):
    """Trim empty border so the knee fills more of the frame."""
    m = vol_u8.max(axis=0) > 8
    ys, xs = np.where(m)
    if len(ys) == 0:
        return vol_u8
    y0, y1, x0, x1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1
    y0, x0 = max(0, y0-4), max(0, x0-4)
    y1, x1 = min(vol_u8.shape[1], y1+4), min(vol_u8.shape[2], x1+4)
    return vol_u8[:, y0:y1, x0:x1]

def load_series(series_dir, laterality):
    """One series -> (K, 3, IMG, IMG), canonicalised to a right knee."""
    files = sorted(glob.glob(f'{series_dir}/*.dcm'))
    if not files:
        return None, None
    heads = [pydicom.dcmread(f, stop_before_pixels=True) for f in files]
    iop = next(h.ImageOrientationPatient for h in heads if hasattr(h, 'ImageOrientationPatient'))
    plane, row, col = plane_from_iop(iop)

    # physical slice order, not InstanceNumber
    nrm = np.cross(row, col)
    pos = np.array([np.dot(np.array(h.ImagePositionPatient, float), nrm) for h in heads])
    order = np.argsort(pos)
    if plane == 'sagittal':
        # slice 0 := most medial, whichever knee this is
        xs = np.array([center_x(h) for h in heads])
        order = np.argsort(xs)[::-1] if laterality == 'R' else np.argsort(xs)

    # decode only the K slices we will actually use
    sel = np.linspace(0, len(files) - 1, K).round().astype(int)
    idx = [order[i] for i in sel]
    vol = np.stack([pydicom.dcmread(files[i]).pixel_array for i in idx]).astype(np.float32)

    # percentiles over the SERIES, not per slice: preserves inter-slice contrast
    lo, hi = np.percentile(vol, [0.5, 99.5])
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    vol = (vol * 255).astype(np.uint8)
    vol = crop_background(vol)

    if plane in ('coronal', 'axial') and laterality == 'L':
        vol = vol[:, :, ::-1]

    out = np.empty((K, IMG, IMG), np.float32)
    for j in range(K):
        out[j] = cv2.resize(vol[j], (IMG, IMG), interpolation=cv2.INTER_AREA)
    out /= 255.0
    out = out[:, None].repeat(3, axis=1)
    out = (out - IM_MEAN[None,:,None,None]) / IM_STD[None,:,None,None]
    return torch.from_numpy(np.ascontiguousarray(out)), PLANES[plane]

## 4. From 176 slices to twelve numbers

A study averages 5.5 series of ~34 slices. The head reduces that to twelve probabilities in two stages.

**Slices → series: attention pooling.** Mean pooling is the honest baseline, but a meniscal tear lives in three slices out of thirty and averaging dilutes it. A learned scalar score per slice, softmaxed across the series, lets the model concentrate where the finding is. Two extras are added to each slice embedding before scoring:

- **normalised slice position**, which after canonicalisation is *anatomy*: 0 is medial, 1 is lateral. The model gets the medial/lateral distinction — four labels' worth — almost for free.
- **a plane embedding**, so one pooling module serves sagittal, coronal and axial without pretending they are interchangeable.

**Series → study: mean within plane, then concatenate.** Series counts are unbalanced (9 864 sagittal / 8 609 coronal / 5 898 axial), so pooling across series uniformly would over-weight sagittal. Averaging within each plane and concatenating the three gives every plane one vote regardless of how many series it contributed; a missing plane contributes a zero vector.

**Features.** Each slice contributes both the CLS token and the mean of the patch tokens. CLS summarises globally; patch-mean retains a little more local structure. Using both beat either alone on OOF and on the 58, and costs nothing at inference since both fall out of the same forward pass.

**On the backbone choice.** DINOv2 is self-supervised — trained to agree with itself across augmented views, never told what an object is. That makes its features unusually good under linear probing, which is what made the frozen baseline viable in an afternoon. It is Apache 2.0, so it redistributes cleanly as a Kaggle dataset, which matters when internet is off. DINOv3 is stronger and its patch-16 grid is tidier, but its weights are gated behind an approval process; not worth the licence risk for a couple of points.

One implementation note that also explains how Stage B was built. `timm` hosts DINOv2 at 518px, and passing `img_size=N` makes it bicubically resample the position embeddings once at load — standard procedure, since attention and MLPs are agnostic to token count and DINOv2 was trained multi-crop. The same mechanism carries a Stage A checkpoint (16×16 grid) into a Stage B model (24×24), which is why Stage B could start from adapted weights instead of from scratch.

In [ ]:
class Head(nn.Module):
    """Attention pooling over slices, mean-within-plane over series, MLP to 12 logits."""
    def __init__(self, d_in, d=256, K=32, n_cls=12, p_drop=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, d))
        self.plane_emb = nn.Embedding(3, d)
        self.register_buffer('pos', torch.linspace(0, 1, K).view(1, 1, K, 1))
        self.pos_proj = nn.Linear(1, d)
        self.att = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
        self.mlp = nn.Sequential(nn.LayerNorm(3 * d), nn.Dropout(p_drop),
                                 nn.Linear(3 * d, d), nn.GELU(),
                                 nn.Dropout(p_drop), nn.Linear(d, n_cls))

    def forward(self, x, pl, mask):                 # x (B,S,K,d_in)
        h = self.proj(x) + self.pos_proj(self.pos) + self.plane_emb(pl)[:, :, None]
        a = self.att(h).softmax(dim=2)              # attention across slices
        sv = (a * h).sum(dim=2)                     # one vector per series
        outs = []
        for p in range(3):                          # mean within each plane
            sel = (pl == p) & mask
            n = sel.sum(1, keepdim=True).clamp(min=1)
            outs.append((sv * sel[..., None]).sum(1) / n)
        return self.mlp(torch.cat(outs, dim=1))

# pretrained=False: weights come from the checkpoints, not from the network
models = []
for f in FOLDS:
    ck = torch.load(f'{D_FT}/ft_f{f}_best.pt', map_location='cuda')
    bb = timm.create_model('vit_base_patch14_reg4_dinov2', pretrained=False,
                           num_classes=0, img_size=IMG).cuda().eval()
    bb.load_state_dict(ck['backbone'])
    d_in = bb.num_features * (2 if FEAT == 'both' else 1)
    hd = Head(d_in, K=K).cuda().eval()
    hd.load_state_dict(ck['head'])
    models.append((bb, hd))
    print(f"fold {f}: OOF {ck.get('oof_macro', float('nan')):.4f}")
NPREFIX = models[0][0].num_prefix_tokens          # 1 CLS + 4 registers
print(len(models), 'models |', round(torch.cuda.memory_allocated()/1e9, 2), 'GB')

## 5. Inference: decode once, score five times

The loop below is ordered deliberately. Each study is decoded **once** and the resulting tensors — already on the GPU — are passed through all five models.

That ordering is what makes a five-model ensemble affordable. DICOM decoding is CPU-bound and dominates the budget; the backbone is a smaller share. Looping models on the outside and re-reading pixels per model would multiply the expensive part by five instead of the cheap one. It is also why Stage B costs only about an hour more than Stage A despite 2.25× the backbone FLOPs — the shared half of the work does not scale with resolution.

**Per-study fallback.** Any study that raises is scored at training prevalence rather than aborting the run. A submission with three fallback studies loses a rounding error; a notebook that dies at study 900 loses the day. Failures are printed at the end so a systematic problem is visible rather than buried.

In [ ]:
@torch.no_grad()
def predict_study(study_dir):
    series_dirs = sorted(glob.glob(f'{study_dir}/*'))

    # one header per series is enough to settle laterality for the whole study
    first_heads = []
    for sd in series_dirs:
        f = glob.glob(f'{sd}/*.dcm')
        if f:
            first_heads.append(pydicom.dcmread(f[0], stop_before_pixels=True))
    lat = study_laterality(first_heads)

    tensors, planes = [], []
    for sd in series_dirs:
        x, pl = load_series(sd, lat)
        if x is not None:
            tensors.append(x.cuda()); planes.append(pl)
    if not tensors:
        raise RuntimeError('no readable series')
    pl = torch.tensor(planes, device='cuda')[None]
    m = torch.ones_like(pl, dtype=torch.bool)

    probs = []
    for bb, hd in models:                       # pixels decoded once, reused here
        embs = []
        for x in tensors:
            with torch.autocast('cuda', dtype=torch.float16):
                ft = bb.forward_features(x)
            cls, pool = ft[:, 0], ft[:, NPREFIX:].mean(1)
            e = {'cls': cls, 'pool': pool}.get(FEAT, torch.cat([cls, pool], 1))
            embs.append(e.float())
        probs.append(torch.sigmoid(hd(torch.stack(embs)[None], pl, m)))
    return torch.stack(probs).mean(0)[0].cpu().numpy()

rows, fails = [], []
for i, st in enumerate(test_studies):
    try:
        p = predict_study(f'{TEST_DIR}/{st}')
    except Exception as e:
        fails.append((st, repr(e)))
        p = np.array([PREV[c] for c in CLASSES])      # fallback: prevalence
    rows.append([st, *p])
    if i % 100 == 0:
        el = time.time() - T0
        print(f'{i:>5}/{len(test_studies)}  {el/60:6.1f} min  '
              f'ETA {el/max(i,1)*len(test_studies)/60:6.1f} min', flush=True)

print(f'\nfailures: {len(fails)}')
for st, e in fails[:10]:
    print(' ', st, e)

## 6. Submission contract

Row order and column names come from `sample_submission.csv` by construction rather than by convention, and three assertions fire before anything is written. Cheap insurance against the failure mode where a run succeeds, scores zero, and the reason is a column order.

In [ ]:
sub = pd.DataFrame(rows, columns=['StudyInstanceUID', *CLASSES])
sample = pd.read_csv(f'{COMP}/sample_submission.csv')
sub = sample[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
assert list(sub.columns) == list(sample.columns), 'column mismatch'
assert sub[CLASSES].notna().all().all(), 'missing predictions'
assert ((sub[CLASSES] >= 0) & (sub[CLASSES] <= 1)).all().all(), 'out of range'
sub.to_csv('submission.csv', index=False)
print(sub.shape, f'| stage {STAGE} | folds {FOLDS} | runtime {(time.time()-T0)/60:.1f} min')
sub.head()

---

## 7. What was learned, including the decision I got wrong

### Domain adaptation does most of the work

The frozen baseline's error profile was legible: diffuse findings scored well (medial OA 0.88, Baker's 0.90, effusion 0.87 on the 58) while focal ones collapsed (medial meniscus 0.65, fracture 0.66, MCL 0.68). The obvious reading is that 224px cannot resolve a structure twenty pixels across, and the obvious fix is more pixels.

That reading was mostly wrong. Fine-tuning at the **same 224px** moved exactly the focal findings the most:

| finding | frozen | fine-tuned @224 | Δ |
|---|---:|---:|---:|
| Medial Meniscus | 0.679 | 0.850 | **+0.171** |
| MCL | 0.708 | 0.825 | **+0.118** |
| ACL | 0.727 | 0.840 | **+0.113** |
| Contusion | 0.676 | 0.775 | +0.099 |

A backbone trained on natural images does not know *what to look for* in an MRI. Once it does, 224px is enough to find a meniscal tear. Resolution was a real but secondary axis — worth +0.017 on top, not the +0.09 that adaptation bought.

### The mistake: gating on the wrong metric

Stage B was very nearly abandoned. The gate was a single fold's OOF macro AUC, with a threshold set in advance at +0.010. Stage B returned **+0.0035** and flat curves across its last three epochs. By the stated rule, archive it.

Run in full, it scored **0.883** — seventeen thousandths above Stage A, from a signal the gate read as four.

The amplification is not luck, it is structural. OOF is measured against report-derived pseudo-labels, and those have a ceiling near 0.88–0.90 because report and image genuinely disagree. When the model gets *better at seeing the knee*, it departs from the labels precisely on the studies where the report was wrong. A real vision improvement is therefore partly booked as disagreement with the teacher. **The local metric systematically understates exactly the kind of gain that matters**, and the effect grows as the model approaches the teacher's ceiling.

The corrected protocol, for anyone building on this:

- **OOF selects epochs and detects breakage.** It has n = 4 407 and low variance; it is the right tool for "did this run go wrong".
- **The gold-58 set decides whether a direction is worth pursuing.** It is noisy — bootstrap intervals run about ±0.04 — but it measures against the same ground truth the leaderboard does. When the two disagree, that is not a tie broken by sample size; they are measuring different things, and only one of them is measuring the target.

I had the rule backwards, and it cost a near-miss on the better model.

### The 58 studies predict the leaderboard

Both scored systems landed **0.044 above** their gold-58 macro AUC (0.771 → 0.776; 0.824 → 0.866). A published external system with a different architecture entirely reports 0.857 gold-58 against 0.903 LB — the same offset to within a thousandth. Three independent systems, one constant.

That turns a 58-study local check into a leaderboard estimator and makes submissions a confirmation step rather than an exploration tool. Whatever your local anchor is, it is worth measuring its offset before spending submissions to find one.

### Fracture is a label problem, not a vision problem

Fracture sits at 0.635 and moved +0.009 under fine-tuning. About 97% of studies share one value for this class — the prior assigned when the report simply does not mention fractures. A target with almost no variance produces almost no gradient, and no amount of modelling recovers it. Fixing this means changing how the label is derived.

### Diversity is a live axis

The 224px and 336px models correlate at **0.901** — meaningfully different despite near-identical local accuracy — and averaging their fold-0 predictions gained +0.006 over either alone. Cheaper sources of the same effect (seeds, unfreeze depth, a different architecture family) are the obvious next step, and a rank-mean rather than a probability-mean is the right way to combine them, since AUC reads order and averaging sigmoids lets the most confident member dominate.

### Efficiency arithmetic

The efficiency prize weights `AUC/(0.5 − maxAUC) + RuntimeSeconds/32400`, which works out to roughly **+0.044 AUC needed per extra hour** to break even. Stage B buys +0.017 for an extra hour, so:

- **Main leaderboard → `STAGE = 'B'`.** More AUC, and 4 h is well inside the 9 h budget.
- **Efficiency prize → `STAGE = 'A'`.** The hour saved is worth more than the seventeen thousandths given up.

Dropping folds is the other lever, and a worse one: five folds cost ~1 h over one fold and are worth more than the 0.044 that trade would require. Runtime is better bought back on the CPU side by parallelising DICOM decode, which costs no accuracy at all.

---

*Every number above is either measured in this notebook's training half or is a completed Kaggle row. No external predictions are blended into either score.*

*If the reasoning was useful — particularly the part where the decision rule was wrong — an upvote helps other competitors find it.*